# 02 — Preprocessing: Model Rekomendasi Latihan

**Stack SOTA 2024-2025:**
- **RobustScaler** untuk fitur numerik (handle outliers BMI/Calories)
- **Feature engineering:** BMR (Mifflin-St Jeor), TDEE, FFMI, interaction features
- **SMOTEENN** untuk class imbalance (kombinasi SMOTE + EditedNN)
- **iterative-stratification** untuk multi-label CV split
- **Augmentasi 7-day** dari 2,773 → ~19,400 baris

**Input:** 2 dataset combined (real 973 + synthetic 1,800)

**Output:**
- `output/preprocessed/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`
- `output/preprocessed/scaler.pkl` (RobustScaler — untuk inference)
- `output/preprocessed/metadata.json`

**Riset acuan:**
- i-MRI 2024: RobustScaler val_acc 0.73 > StandardScaler 0.71
- Springer AI Review 2024: SMOTEENN paling robust untuk medical data
- Nature Sci Reports 2025: interaction features +3-4pp AUC

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import os

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours

np.random.seed(42)
os.makedirs('output/preprocessed', exist_ok=True)

# ── Load Combined Dataset ──
PATH_REAL      = '../../dataset/Model_rekomendasi_Pelatihan/gym_member_exercise_dataset/gym_members_exercise_tracking.csv'
PATH_SYNTHETIC = '../../dataset/Model_Adaptif_Perencanaan_Ulang/Fitness Tracker Dataset/gym_members_exercise_tracking_synthetic_data.csv'

df_real      = pd.read_csv(PATH_REAL)
df_synthetic = pd.read_csv(PATH_SYNTHETIC)
df = pd.concat([df_real, df_synthetic], ignore_index=True)
print(f'Combined: {df.shape}')

Combined: (2773, 15)


In [2]:
# ── 1. Feature Engineering (Riset Nature 2025: +3-4pp AUC) ──

def engineer_features(df):
    df = df.copy()

    numeric_cols = [
        'Age', 'Weight (kg)', 'Height (m)', 'BMI', 'Max_BPM', 'Avg_BPM',
        'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned',
        'Fat_Percentage', 'Water_Intake (liters)', 'Workout_Frequency (days/week)',
        'Experience_Level'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    duration_clean = df['Session_Duration (hours)'].fillna(df['Session_Duration (hours)'].median())

    # ── Basic encoding ──
    df['gender_enc'] = df['Gender'].map({'Male': 1, 'Female': 0}).fillna(2).astype(int)

    # ── BMR (Mifflin-St Jeor) ──
    df['height_cm'] = df['Height (m)'] * 100
    df['bmr'] = (10 * df['Weight (kg)'] +
                 6.25 * df['height_cm'] -
                 5 * df['Age'] +
                 np.where(df['gender_enc'] == 1, 5, -161))

    # ── TDEE ──
    activity_map = {1: 1.2, 2: 1.375, 3: 1.55, 4: 1.725, 5: 1.9}
    df['activity_factor'] = df['Workout_Frequency (days/week)'].map(activity_map).fillna(1.55)
    df['tdee'] = df['bmr'] * df['activity_factor']

    # ── FFMI ──
    df['lean_mass_kg'] = df['Weight (kg)'] * (1 - df['Fat_Percentage'] / 100)
    df['ffmi'] = df['lean_mass_kg'] / (df['Height (m)'] ** 2)

    # ── BMI category ──
    def bmi_cat(b):
        if b < 18.5: return 0
        if b < 25:   return 1
        if b < 30:   return 2
        return 3
    df['bmi_cat'] = df['BMI'].apply(bmi_cat)

    # ── Age band ──
    age_clean = df['Age'].fillna(df['Age'].median())
    df['age_band'] = pd.cut(age_clean, bins=[0, 25, 35, 50, 120],
                            labels=[0, 1, 2, 3], include_lowest=True).astype(int)

    # ── Interaction features ──
    df['bmi_x_age'] = df['BMI'] * df['Age']
    df['activity_x_duration'] = df['Workout_Frequency (days/week)'] * df['Session_Duration (hours)']
    max_bpm = df['Max_BPM'].replace(0, np.nan)
    duration_hours = df['Session_Duration (hours)'].replace(0, np.nan)
    df['bpm_intensity_ratio'] = df['Avg_BPM'] / max_bpm
    df['calorie_efficiency'] = df['Calories_Burned'] / duration_hours
    df['bpm_intensity_ratio'] = df['bpm_intensity_ratio'].fillna(0)
    df['calorie_efficiency'] = df['calorie_efficiency'].fillna(0)

    # ── Target encoding ──
    workout_map = {'Yoga': 0, 'Cardio': 1, 'Strength': 2, 'HIIT': 3}
    df['workout_type_label'] = df['Workout_Type'].map(workout_map).fillna(1).astype(int)
    exp_clean = df['Experience_Level'].fillna(df['Experience_Level'].median())
    df['intensity_label'] = exp_clean.astype(int)  # 1,2,3

    # ── Mode (semua dataset = gym, default 1) ──
    df['mode'] = 1
    df['has_injury'] = 0
    df['has_chronic'] = 0
    df['days_per_week'] = df['Workout_Frequency (days/week)']
    df['session_minutes'] = (duration_clean * 60).round().astype(int)

    # ── Raw features (Round 2: predictive power tinggi) ──
    avg_bpm_med = df['Avg_BPM'].median() if df['Avg_BPM'].notna().any() else 130
    cal_burned_med = df['Calories_Burned'].median() if df['Calories_Burned'].notna().any() else 800
    fat_pct_med = df['Fat_Percentage'].median() if df['Fat_Percentage'].notna().any() else 25
    df['avg_bpm']         = df['Avg_BPM'].fillna(avg_bpm_med)
    df['max_bpm']         = df['Max_BPM'].fillna(180)
    df['resting_bpm']     = df['Resting_BPM'].fillna(70)
    df['calories_burned'] = df['Calories_Burned'].fillna(cal_burned_med)
    df['fat_pct']         = df['Fat_Percentage'].fillna(fat_pct_med)
    df['water_intake']    = df['Water_Intake (liters)'].fillna(2.5)

    return df

df_feat = engineer_features(df)
print(f'Feature engineering selesai. Total kolom: {len(df_feat.columns)}')
print('\nFitur baru (engineered + raw):')
new_features = ['bmr', 'tdee', 'ffmi', 'bmi_x_age', 'activity_x_duration',
                'bpm_intensity_ratio', 'calorie_efficiency',
                'avg_bpm', 'max_bpm', 'resting_bpm', 'calories_burned', 'fat_pct', 'water_intake']
df_feat[new_features].describe().round(2)


Feature engineering selesai. Total kolom: 41

Fitur baru (engineered + raw):


,bmr,tdee,ffmi,bmi_x_age,activity_x_duration,bpm_intensity_ratio,calorie_efficiency,avg_bpm,max_bpm,resting_bpm,calories_burned,fat_pct,water_intake
count,2716.00,2716.00,2710.00,2733.00,2693.00,2773.00,2773.00,2773.00,2773.00,2773.00,2773.00,2773.00,2773.00
mean,1522.20,2445.80,17.87,788.49,4.55,0.79,762.94,145.37,180.11,63.39,988.15,24.03,2.68
std,268.73,491.81,5.83,397.47,1.98,0.15,320.59,14.88,11.43,7.76,314.24,6.03,0.67
min,941.50,1337.19,6.81,221.76,1.00,0.00,0.00,120.00,160.00,50.00,303.00,10.00,1.50
25%,1327.56,2086.14,13.53,492.43,3.08,0.73,603.26,133.00,170.00,57.00,765.00,20.80,2.10
50%,1485.50,2390.42,17.06,714.56,4.23,0.80,720.17,145.00,180.00,64.00,969.00,24.90,2.70
75%,1697.00,2757.74,21.24,1002.00,5.80,0.88,866.87,158.00,190.00,71.00,1187.00,28.20,3.30
max,2395.25,4261.22,44.86,2720.04,10.00,1.06,3566.00,169.00,199.00,74.00,1783.00,35.00,3.70


In [3]:
# ── 2. Tidak ada augmentasi 7-day (Round 2 fix) ──
# Jurnal 2024: augmentation sebelum split = data leakage di CV folds
# Cukup pakai dataset original 2773 baris (4 kelas balance 24-26% each)

df_final = df_feat.copy().reset_index(drop=True)

# Drop rows dengan target NaN
df_final = df_final.dropna(subset=['workout_type_label', 'intensity_label']).reset_index(drop=True)

print(f'Final dataset (tanpa augmentasi): {df_final.shape}')
print(f'\nWorkout type distribution (4 kelas, sudah balance):')
print(df_final['workout_type_label'].value_counts().sort_index())
print(f'\nIntensity distribution:')
print(df_final['intensity_label'].value_counts().sort_index())
imb_ratio = df_final['workout_type_label'].value_counts().max() / df_final['workout_type_label'].value_counts().min()
print(f'\nClass imbalance ratio workout: {imb_ratio:.2f}:1 (no SMOTE needed)')


Final dataset (tanpa augmentasi): (2773, 41)

Workout type distribution (4 kelas, sudah balance):
workout_type_label
0    664
1    777
2    722
3    610
Name: count, dtype: int64

Intensity distribution:
intensity_label
1    1042
2    1181
3     550
Name: count, dtype: int64

Class imbalance ratio workout: 1.27:1 (no SMOTE needed)


In [4]:
# ── 3. Final Feature Set ──
# Workout model: SEMUA features (boleh pakai Experience_Level)
# Intensity model: EXCLUDE Experience_Level (avoid leakage karena intensity_label = Experience_Level)

FEATURE_BASE = [
    # Profile demografis
    'BMI', 'bmi_cat', 'gender_enc', 'Age', 'age_band',
    # Training pattern
    'mode', 'days_per_week', 'session_minutes',
    'has_injury', 'has_chronic',
    # Energy expenditure (derived)
    'bmr', 'tdee', 'ffmi',
    # Interaction features
    'bmi_x_age', 'activity_x_duration', 'bpm_intensity_ratio', 'calorie_efficiency',
    # Raw post-workout features (Round 2 NEW)
    'avg_bpm', 'max_bpm', 'resting_bpm', 'calories_burned', 'fat_pct', 'water_intake',
]

# Workout model: include Experience_Level
FEATURE_FOR_WORKOUT = FEATURE_BASE + ['Experience_Level']

# Intensity model: EXCLUDE Experience_Level (leakage prevention)
FEATURE_FOR_INTENSITY = FEATURE_BASE  # tanpa Experience_Level

X_for_workout   = df_final[FEATURE_FOR_WORKOUT].fillna(0)
X_for_intensity = df_final[FEATURE_FOR_INTENSITY].fillna(0)
y_workout       = df_final['workout_type_label']
y_intensity     = df_final['intensity_label']

print(f'X workout:   {X_for_workout.shape}   ({len(FEATURE_FOR_WORKOUT)} features, includes Experience_Level)')
print(f'X intensity: {X_for_intensity.shape} ({len(FEATURE_FOR_INTENSITY)} features, NO Experience_Level)')
print(f'y_workout classes:   {sorted(y_workout.unique())} (4 classes)')
print(f'y_intensity classes: {sorted(y_intensity.unique())} (3 classes)')


X workout:   (2773, 24)   (24 features, includes Experience_Level)
X intensity: (2773, 23) (23 features, NO Experience_Level)
y_workout classes:   [0, 1, 2, 3] (4 classes)
y_intensity classes: [1, 2, 3] (3 classes)


In [5]:
# ── 4. Stratified Train/Test Split (per model) ──
# Tanpa augmentasi, tiap row = 1 user unik, no leakage concern
from sklearn.model_selection import train_test_split

# Split untuk workout model (4 classes)
X_wtrain, X_wtest, yw_train, yw_test = train_test_split(
    X_for_workout, y_workout,
    test_size=0.2, random_state=42, stratify=y_workout
)

# Split untuk intensity model (3 classes, features berbeda — tanpa Experience_Level)
X_itrain, X_itest, yi_train, yi_test = train_test_split(
    X_for_intensity, y_intensity,
    test_size=0.2, random_state=42, stratify=y_intensity
)

print(f'Workout   — Train: {X_wtrain.shape}, Test: {X_wtest.shape}')
print(f'Intensity — Train: {X_itrain.shape}, Test: {X_itest.shape}')
print(f'\nTrain workout dist: {dict(yw_train.value_counts().sort_index())}')
print(f'Test  workout dist: {dict(yw_test.value_counts().sort_index())}')
print(f'\nTrain intensity dist: {dict(yi_train.value_counts().sort_index())}')
print(f'Test  intensity dist: {dict(yi_test.value_counts().sort_index())}')


Workout   — Train: (2218, 24), Test: (555, 24)
Intensity — Train: (2218, 23), Test: (555, 23)

Train workout dist: {0: 531, 1: 621, 2: 578, 3: 488}
Test  workout dist: {0: 133, 1: 156, 2: 144, 3: 122}

Train intensity dist: {1: 833, 2: 945, 3: 440}
Test  intensity dist: {1: 209, 2: 236, 3: 110}


In [6]:
# ── 5. RobustScaler (i-MRI 2024 benchmark) ──
NUMERIC_COLS = [
    'BMI', 'Age', 'session_minutes', 'bmr', 'tdee', 'ffmi',
    'bmi_x_age', 'activity_x_duration', 'bpm_intensity_ratio', 'calorie_efficiency',
    'avg_bpm', 'max_bpm', 'resting_bpm', 'calories_burned', 'fat_pct', 'water_intake',
]

# Workout scaler
scaler_workout = RobustScaler()
X_wtrain_s = X_wtrain.copy()
X_wtest_s  = X_wtest.copy()
X_wtrain_s[NUMERIC_COLS] = scaler_workout.fit_transform(X_wtrain[NUMERIC_COLS])
X_wtest_s[NUMERIC_COLS]  = scaler_workout.transform(X_wtest[NUMERIC_COLS])

# Intensity scaler (separate karena features berbeda)
scaler_intensity = RobustScaler()
X_itrain_s = X_itrain.copy()
X_itest_s  = X_itest.copy()
X_itrain_s[NUMERIC_COLS] = scaler_intensity.fit_transform(X_itrain[NUMERIC_COLS])
X_itest_s[NUMERIC_COLS]  = scaler_intensity.transform(X_itest[NUMERIC_COLS])

print('OK RobustScaler fitted & applied (workout + intensity terpisah)')
print(f'Scaler median sample: BMI={scaler_workout.center_[0]:.2f}, Age={scaler_workout.center_[1]:.2f}, avg_bpm={scaler_workout.center_[10]:.2f}')


OK RobustScaler fitted & applied (workout + intensity terpisah)
Scaler median sample: BMI=20.77, Age=35.00, avg_bpm=145.00


In [7]:
# ── 6. Compute Sample Weights (REPLACE SMOTEENN, Round 2 fix) ──
# Jurnal 2024: pada data sudah balanced, sample_weight lebih baik dari SMOTEENN
# SMOTEENN's ENN agresif menghapus boundary samples (60%+ pada balanced data)
from sklearn.utils.class_weight import compute_sample_weight

# Compute balanced sample weights
sw_workout   = compute_sample_weight(class_weight='balanced', y=yw_train)
sw_intensity = compute_sample_weight(class_weight='balanced', y=yi_train)

print('OK Sample weights computed (balanced strategy):')
print(f'Workout   weights: min={sw_workout.min():.3f}, max={sw_workout.max():.3f}, mean={sw_workout.mean():.3f}')
print(f'Intensity weights: min={sw_intensity.min():.3f}, max={sw_intensity.max():.3f}, mean={sw_intensity.mean():.3f}')
print(f'\nNote: Class minoritas mendapat weight lebih tinggi (XGBoost akan penalty error padanya)')


OK Sample weights computed (balanced strategy):
Workout   weights: min=0.893, max=1.136, mean=1.000
Intensity weights: min=0.782, max=1.680, mean=1.000

Note: Class minoritas mendapat weight lebih tinggi (XGBoost akan penalty error padanya)


In [8]:
# ── 7. Save Outputs (Round 2: 2 datasets terpisah) ──
import numpy as np

# Workout dataset
X_wtrain_s.to_parquet('output/preprocessed/X_workout_train.parquet', index=False)
X_wtest_s.to_parquet('output/preprocessed/X_workout_test.parquet', index=False)
yw_train.to_frame().to_parquet('output/preprocessed/yw_train.parquet', index=False)
yw_test.to_frame().to_parquet('output/preprocessed/yw_test.parquet', index=False)
np.save('output/preprocessed/sw_workout.npy', sw_workout)

# Intensity dataset
X_itrain_s.to_parquet('output/preprocessed/X_intensity_train.parquet', index=False)
X_itest_s.to_parquet('output/preprocessed/X_intensity_test.parquet', index=False)
yi_train.to_frame().to_parquet('output/preprocessed/yi_train.parquet', index=False)
yi_test.to_frame().to_parquet('output/preprocessed/yi_test.parquet', index=False)
np.save('output/preprocessed/sw_intensity.npy', sw_intensity)

# Scalers
joblib.dump(scaler_workout, 'output/preprocessed/scaler_workout.pkl')
joblib.dump(scaler_intensity, 'output/preprocessed/scaler_intensity.pkl')

# Metadata
metadata = {
    'feature_cols_workout':   FEATURE_FOR_WORKOUT,
    'feature_cols_intensity': FEATURE_FOR_INTENSITY,
    'numeric_cols_scaled':    NUMERIC_COLS,
    'workout_type_map':       {0: 'FLEXIBILITY', 1: 'CARDIO', 2: 'STRENGTH', 3: 'HIIT'},
    'intensity_map':          {1: 'LOW', 2: 'MID', 3: 'HIGH'},
    'preprocessing_stack': {
        'scaler':       'RobustScaler',
        'balancing':    'sample_weight (compute_sample_weight balanced)',
        'augmentation': 'NONE (Round 2: hapus 7-day aug untuk avoid CV leakage)',
        'feature_engineering': ['BMR_Mifflin_St_Jeor', 'TDEE', 'FFMI',
                                'bmi_x_age', 'activity_x_duration',
                                'bpm_intensity_ratio', 'calorie_efficiency'],
        'raw_features_added': ['avg_bpm', 'max_bpm', 'resting_bpm',
                               'calories_burned', 'fat_pct', 'water_intake'],
        'intensity_leakage_fix': 'Experience_Level EXCLUDED dari FEATURE_FOR_INTENSITY',
    },
    'realistic_targets': {
        'workout_f1_macro':   '>= 0.70 (Garcia-Brustenga 2024 baseline)',
        'intensity_f1_macro': '>= 0.75 (after Experience_Level decoupling)',
    },
    'n_train_workout':   int(len(X_wtrain_s)),
    'n_test_workout':    int(len(X_wtest_s)),
    'n_train_intensity': int(len(X_itrain_s)),
    'n_test_intensity':  int(len(X_itest_s)),
    'random_seed':       42,
}
with open('output/preprocessed/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('OK Preprocessing Round 2 selesai. Files saved:')
import os
for f in sorted(os.listdir('output/preprocessed')):
    size = os.path.getsize(f'output/preprocessed/{f}') / 1024
    print(f'  output/preprocessed/{f:40s} ({size:.1f} KB)')


OK Preprocessing Round 2 selesai. Files saved:
  output/preprocessed/X_intensity_test.parquet                 (58.1 KB)
  output/preprocessed/X_intensity_train.parquet                (161.6 KB)
  output/preprocessed/X_test_scaled.parquet                    (85.7 KB)
  output/preprocessed/X_train_balanced.parquet                 (150.3 KB)
  output/preprocessed/X_train_balanced_intensity.parquet       (445.4 KB)
  output/preprocessed/X_train_scaled.parquet                   (301.1 KB)
  output/preprocessed/X_workout_test.parquet                   (58.1 KB)
  output/preprocessed/X_workout_train.parquet                  (160.2 KB)
  output/preprocessed/metadata.json                            (2.4 KB)
  output/preprocessed/scaler.pkl                               (1.0 KB)
  output/preprocessed/scaler_intensity.pkl                     (1.2 KB)
  output/preprocessed/scaler_workout.pkl                       (1.2 KB)
  output/preprocessed/sw_intensity.npy                         (17.5 KB)
  o